# A/B Testing: Optimizing Marketing Strategies

# Import Libraries

In [1]:
import pandas as pd
import datetime
from datetime import date, timedelta
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Read Data

In [2]:
control_data = pd.read_csv("/kaggle/input/ab-testing-dataset/control_group.csv", sep = ";")
test_data = pd.read_csv("/kaggle/input/ab-testing-dataset/test_group.csv", sep = ";")
print(control_data.head())
print('-' * 100)
print(test_data.head())

      Campaign Name       Date  Spend [USD]  # of Impressions     Reach  \
0  Control Campaign  1.08.2019         2280           82702.0   56930.0   
1  Control Campaign  2.08.2019         1757          121040.0  102513.0   
2  Control Campaign  3.08.2019         2343          131711.0  110862.0   
3  Control Campaign  4.08.2019         1940           72878.0   61235.0   
4  Control Campaign  5.08.2019         1835               NaN       NaN   

   # of Website Clicks  # of Searches  # of View Content  # of Add to Cart  \
0               7016.0         2290.0             2159.0            1819.0   
1               8110.0         2033.0             1841.0            1219.0   
2               6508.0         1737.0             1549.0            1134.0   
3               3065.0         1042.0              982.0            1183.0   
4                  NaN            NaN                NaN               NaN   

   # of Purchase  
0          618.0  
1          511.0  
2          372.0  
3   

# Data Preparation

The datasets have some errors in column names. Let’s give new column names before moving forward:

In [3]:
control_data.columns = ["Campaign Name", "Date", "Amount Spent", 
                        "Number of Impressions", "Reach", "Website Clicks", 
                        "Searches Received", "Content Viewed", "Added to Cart",
                        "Purchases"]

test_data.columns = ["Campaign Name", "Date", "Amount Spent", 
                        "Number of Impressions", "Reach", "Website Clicks", 
                        "Searches Received", "Content Viewed", "Added to Cart",
                        "Purchases"]

# Values Check

Let’s see if the datasets have null values or not:

In [4]:
null_rows = control_data[control_data.isnull().any(axis=1)]
null_row_indices = control_data[control_data.isnull().any(axis=1)].index

print("Rows with missing values:")
print(null_rows)
print("\nRow indices with missing values:", null_row_indices.tolist())

Rows with missing values:
      Campaign Name       Date  Amount Spent  Number of Impressions  Reach  \
4  Control Campaign  5.08.2019          1835                    NaN    NaN   

   Website Clicks  Searches Received  Content Viewed  Added to Cart  Purchases  
4             NaN                NaN             NaN            NaN        NaN  

Row indices with missing values: [4]


Only one row has missing values, with only amount spend data filled, it should be some error while data import or some separator problems, such row should be deleted or the data extraction proccess should be checked. Each campaign could have it's unique nature and could not be filled with mean/median value, it's better to drop such rows.

In [5]:
control_data = control_data.dropna()
print("\nShape of control_data after dropping missing values:", control_data.shape)


Shape of control_data after dropping missing values: (29, 10)


In [6]:
print(test_data.isnull().sum())

Campaign Name            0
Date                     0
Amount Spent             0
Number of Impressions    0
Reach                    0
Website Clicks           0
Searches Received        0
Content Viewed           0
Added to Cart            0
Purchases                0
dtype: int64


Create Dataframes for simpler further calculations.

In [7]:
control = pd.DataFrame(control_data)
test = pd.DataFrame(test_data)

### Metrics Selection

Further the metrics selection should be done. Basic metrics chosen below represent key stages of the user funnel in a marketing campaign:
- Website Clicks: Measures initial user engagement.
- Searches Received: Indicate interest and interaction with the website.
- Added to Cart: Reflects purchase intent.
- Purchases: Tracks final conversions, the ultimate goal.
- Conversion Rates: Provide insights into efficiency at different funnel stages.
- Cost per Conversion: Evaluates cost-effectiveness, critical for budget optimization.
- Content Viewed: Measures initial user engagement as well as webside clicks.
- CPA = Amount Spent / Purchases (Cost Per Acquisition): Determines profitability of advertising campaigns.
- CTR = Website Clicks / Number of Impressions (Click-Through Rate): Identifies high-performing ad creatives.

These metrics collectively assess campaign performance across awareness, engagement, and conversion, enabling a comprehensive comparison.

In [8]:
metrics = ['Website Clicks', 'Searches Received', 'Added to Cart', 'Purchases', 'Conversion Rate (Purchase)', 'Cost per Conversion', 'Content Viewed', 'CPA', 'CTR']

### Why T-Test Over Z-Test or Other Tests:
- T-Test: Used because it’s suitable for comparing means of two groups (Control vs. Test) with small sample sizes and unknown population variances. The Welch’s t-test (used here with equal_var=False) accounts for unequal variances, making it robust for marketing data where variances often differ.
- Z-Test: Requires large sample sizes and known population variances, which may not apply here due to limited data points (e.g., small number of campaign days) and unknown population parameters.
- Other Tests: Non-parametric tests (e.g., Mann-Whitney U) could be used for non-normal data, but the t-test is appropriate assuming normality or near-normality, which is often reasonable for aggregated campaign metrics. ANOVA or chi-square tests are less relevant since only two groups are compared.

The t-test’s flexibility and applicability to this dataset’s characteristics make it the best choice for detecting significant differences in means.

# Perform A/B testing for each metric

Function to preprocess data and calculate per-row metrics (CPA, CTR).

In [9]:
def preprocess_data(data):
    data = data.copy()
    data['CPA'] = data['Amount Spent'] / data['Purchases']
    data['CPA'] = data['CPA'].replace([np.inf, -np.inf], np.nan).fillna(data['CPA'].mean())
    data['CTR'] = data['Website Clicks'] / data['Number of Impressions']
    data['CTR'] = data['CTR'].replace([np.inf, -np.inf], np.nan).fillna(data['CTR'].mean())
    data['Conversion Rate (Purchase)'] = data['Purchases'] / data['Website Clicks']
    data['Conversion Rate (Purchase)'] = data['Conversion Rate (Purchase)'].replace([np.inf, -np.inf], np.nan).fillna(data['Conversion Rate (Purchase)'].mean())
    data['Cost per Conversion'] = data['Amount Spent'] / data['Purchases']
    data['Cost per Conversion'] = data['Cost per Conversion'].replace([np.inf, -np.inf], np.nan).fillna(data['Cost per Conversion'].mean())
    return data

Function to perform t-test.

In [10]:
def perform_ab_test(control_data, test_data, metric):
    t_stat, p_value = stats.ttest_ind(control_data[metric], test_data[metric], equal_var=False, nan_policy='omit')
    control_mean = control_data[metric].mean()
    test_mean = test_data[metric].mean()
    effect_size = (test_mean - control_mean) / control_data[metric].std() if control_data[metric].std() != 0 else 0
    return {
        'metric': metric,
        'control_mean': control_mean,
        'test_mean': test_mean,
        't_stat': t_stat,
        'p_value': p_value,
        'effect_size': effect_size
    }

Preprocess data to calculate CPA, CTR, and per-row conversion metrics.

In [11]:
control = preprocess_data(control)
test = preprocess_data(test)

Perform t-tests for all metrics.

In [12]:
results = []
for metric in metrics:
    if metric in control.columns:
        result = perform_ab_test(control, test, metric)
        if result:
            results.append(result)

print("A/B Testing Results:")
print("-" * 50)
for result in results:
    print(f"\nMetric: {result['metric']}")
    print(f"Control Mean: {result['control_mean']:.2f}")
    print(f"Test Mean: {result['test_mean']:.2f}")
    print(f"P-value: {result['p_value']:.4f}")
    print(f"Effect Size: {result['effect_size']:.2f}")
    print("Statistically Significant" if result['p_value'] < 0.05 else "Not Statistically Significant")

print("\nConclusions:")
print("-" * 50)
print("Based on the analysis:")
for result in results:
    if result['p_value'] < 0.05:
        print(f"- {result['metric']}: Significant difference found. {'Test' if result['test_mean'] > result['control_mean'] else 'Control'} performs better.")
    else:
        print(f"- {result['metric']}: No significant difference found.")

A/B Testing Results:
--------------------------------------------------

Metric: Website Clicks
Control Mean: 5320.79
Test Mean: 6032.33
P-value: 0.1205
Effect Size: 0.40
Not Statistically Significant

Metric: Searches Received
Control Mean: 2221.31
Test Mean: 2418.97
P-value: 0.2678
Effect Size: 0.23
Not Statistically Significant

Metric: Added to Cart
Control Mean: 1300.00
Test Mean: 881.53
P-value: 0.0001
Effect Size: -1.03
Statistically Significant

Metric: Purchases
Control Mean: 522.79
Test Mean: 521.23
P-value: 0.9760
Effect Size: -0.01
Not Statistically Significant

Metric: Conversion Rate (Purchase)
Control Mean: 0.11
Test Mean: 0.09
P-value: 0.1428
Effect Size: -0.33
Not Statistically Significant

Metric: Cost per Conversion
Control Mean: 5.05
Test Mean: 5.90
P-value: 0.1946
Effect Size: 0.40
Not Statistically Significant

Metric: Content Viewed
Control Mean: 1943.79
Test Mean: 1858.00
P-value: 0.6374
Effect Size: -0.11
Not Statistically Significant

Metric: CPA
Control Mean:

### Recommended Campaign: 
Test Campaign

### Reason
The Test campaign’s significant advantage in CTR and, with a 12.5% significance threshold, in Website Clicks, highlights its superior ability to drive traffic and engagement. Despite the Control’s edge in Added to Cart, the lack of significant differences in Purchases and CPA suggests no substantial trade-off in conversions or cost efficiency. The Test campaign is recommended for its stronger engagement and traffic-driving potential.